# Two-moons usage example

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

from recursive_partition import BaggedRecursivePartitionClassifier, RecursivePartitionClassifier

In [ ]:
X, y = make_moons(n_samples=600, noise=0.22, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

In [ ]:
model = RecursivePartitionClassifier(
    base_estimator=SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced')
)
bagged_model = BaggedRecursivePartitionClassifier(
    estimator=RecursivePartitionClassifier(
        base_estimator=QuadraticDiscriminantAnalysis(priors=[0.5, 0.5], reg_param=0.05)
    ),
    n_estimators=30, n_jobs=-1, random_state=42
)

model.fit(X_train, y_train)
bagged_model.fit(X_train, y_train)
print('RBF SVM accuracy:', model.score(X_test, y_test))
print('Bagged QDA accuracy:', bagged_model.score(X_test, y_test))

In [ ]:
xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 220),
                     np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 220))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for axis, fitted_model, name in zip(axes, [model, bagged_model], ['RBF SVM', 'Bagged QDA']):
    probability = fitted_model.predict_proba(grid)[:, 1].reshape(xx.shape)
    axis.contourf(xx, yy, probability, levels=30, cmap='RdBu_r', alpha=0.55)
    axis.contour(xx, yy, probability, levels=[0.5], colors='black')
    axis.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='RdBu_r', edgecolors='white', linewidths=0.4, s=22)
    axis.set_title(name)
plt.show()